# Post-Harvest Loss Prediction System
**Name:** Aryan Kumar Sinha | **SRN:** PES1PG25CA029  
**Goal:** SDG 2 – Zero Hunger | **Algorithm:** Logistic Regression  
**Objective:** Predict whether a region has **High or Low** post-harvest food loss based on pesticide usage patterns.

In [ ]:
# ── Cell 1: Import Libraries ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

print("Libraries loaded successfully!")

In [ ]:
# ── Cell 2: Load Dataset ──────────────────────────────────────────────────────
# Dataset compiled from FAO / Kaggle / data.gov.in
# Key columns: Area, Item (pesticide type), value (pesticide qty),
#              Year, Sub-region Name, Food loss percentage
df = pd.read_csv("Dataset.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
# ── Cell 3: Explore Dataset ───────────────────────────────────────────────────
print("Columns:", df.columns.tolist())
print()
df.info()

In [ ]:
# ── Cell 4: Check Missing Values ──────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# ── Cell 5: Preprocessing ─────────────────────────────────────────────────────

# Step 1: Keep only 'Agricultural Use' element to avoid duplicate food loss % rows
df = df[df["Element"] == "Agricultural Use"].copy()
print("Rows after filter:", len(df))

# Step 2: Drop rows with missing values in key columns
df.dropna(subset=["value", "Food loss percentage", "Item", "Sub-region Name"], inplace=True)

# Step 3: Remove duplicate rows
df.drop_duplicates(inplace=True)
print("Rows after removing duplicates:", len(df))

# Step 4: Create binary target variable using median threshold
#         High Loss (1) if food loss % >= median  |  Low Loss (0) if below median
median_loss = df["Food loss percentage"].median()
print(f"\nMedian food loss %: {median_loss}")

df["Loss_Class"] = (df["Food loss percentage"] >= median_loss).astype(int)
print("\nClass distribution (0=Low Loss, 1=High Loss):")
print(df["Loss_Class"].value_counts())

In [ ]:
# ── Cell 6: Feature Engineering & Label Encoding ──────────────────────────────
# Logistic Regression needs numeric inputs, so we encode categorical columns

le_item   = LabelEncoder()
le_region = LabelEncoder()

df["Item_enc"]   = le_item.fit_transform(df["Item"])            # pesticide type
df["Region_enc"] = le_region.fit_transform(df["Sub-region Name"])  # geographic region

# Feature matrix X and target y
X = df[["value", "Year", "Item_enc", "Region_enc"]]
y = df["Loss_Class"]

print("Feature matrix shape:", X.shape)
print("Target shape         :", y.shape)

In [ ]:
# ── Cell 7: Train-Test Split ──────────────────────────────────────────────────
# 80% for training, 20% for testing | random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training samples:", X_train.shape[0])
print("Testing  samples:", X_test.shape[0])

In [ ]:
# ── Cell 8: Feature Scaling ───────────────────────────────────────────────────
# Logistic Regression converges better with scaled features (mean=0, std=1)
# Fit scaler ONLY on training data to prevent data leakage

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("Scaling complete.")

In [ ]:
# ── Cell 9: Train Logistic Regression Model ───────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)
print("Model training complete!")

In [ ]:
# ── Cell 10: Model Evaluation ─────────────────────────────────────────────────
y_pred = model.predict(X_test_sc)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy : {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low Loss", "High Loss"]))

In [ ]:
# ── Cell 11: Confusion Matrix ─────────────────────────────────────────────────
# Shows how many predictions were correct (diagonal) vs wrong (off-diagonal)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low Loss", "High Loss"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix – Post-Harvest Loss Prediction")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 12: EDA – Average Food Loss by Sub-region ────────────────────────────
plt.figure(figsize=(10, 5))
region_loss = (df.groupby("Sub-region Name")["Food loss percentage"]
                 .mean()
                 .sort_values(ascending=False))
region_loss.plot(kind="bar", color="tomato", edgecolor="black")
plt.title("Average Food Loss % by Sub-region")
plt.xlabel("Sub-region")
plt.ylabel("Avg Food Loss %")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("loss_by_region.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 13: Feature Importance (Logistic Regression Coefficients) ────────────
# Larger absolute coefficient = stronger influence on prediction
feature_names = ["Pesticide Value", "Year", "Pesticide Type", "Sub-region"]
coefs = pd.Series(np.abs(model.coef_[0]), index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
coefs.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Feature Importance – Absolute Coefficients")
plt.ylabel("Coefficient Magnitude")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

print("\nCoefficients:")
print(coefs)

In [ ]:
# ── Cell 14: Predict on a New Sample ──────────────────────────────────────────
# Demonstrate the model on a custom input (change values to test scenarios)
sample = pd.DataFrame({
    "value"      : [150],   # pesticide quantity (tonnes)
    "Year"       : [2020],  # year of record
    "Item_enc"   : [0],     # encoded pesticide type (0 = first in sorted order)
    "Region_enc" : [2]      # encoded sub-region (2 = third in sorted order)
})

sample_scaled = scaler.transform(sample)
prediction    = model.predict(sample_scaled)
probability   = model.predict_proba(sample_scaled)

label = "HIGH Loss" if prediction[0] == 1 else "LOW Loss"
print(f"Predicted Loss Category : {label}")
print(f"Probability → Low: {probability[0][0]:.2f}  |  High: {probability[0][1]:.2f}")